In [0]:

# Silver processing parameters

dbutils.widgets.text("file_id", "")
dbutils.widgets.text("adf_run_id", "")

file_id = dbutils.widgets.get("file_id").strip()
adf_run_id = dbutils.widgets.get("adf_run_id").strip()

if not file_id or not adf_run_id:
    raise ValueError(
        "file_id and adf_run_id are required"
    )

print("Silver File ID:", file_id)
print("ADF Run ID:", adf_run_id)


In [0]:

ownership_rows = spark.sql("""
    SELECT
        file_id,
        batch_id,
        adf_run_id,
        processing_stage,
        pipeline_status
    FROM dbw_healthcare_de_dev.etl.file_control
    WHERE file_id = :file_id
      AND adf_run_id = :adf_run_id
      AND processing_stage = 'SILVER'
      AND pipeline_status = 'IN_PROGRESS'
      AND is_enabled = TRUE
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

if len(ownership_rows) != 1:
    raise RuntimeError(
        "Silver processing rejected: file is not owned "
        "by the current ADF run or is not in SILVER stage."
    )

batch_id = ownership_rows[0]["batch_id"]

print("Silver ownership validation passed")
print("File ID:", file_id)
print("ADF Run ID:", adf_run_id)
print("Batch ID:", batch_id)


In [0]:

bronze_table = "dbw_healthcare_de_dev.bronze.patients_raw"

bronze_stats = spark.sql("""
    SELECT
        COUNT(*) AS bronze_count,
        COUNT(DISTINCT patient_id) AS unique_patients,
        SUM(
            CASE
                WHEN patient_id IS NULL
                  OR TRIM(patient_id) = ''
                THEN 1 ELSE 0
            END
        ) AS invalid_patient_ids
    FROM dbw_healthcare_de_dev.bronze.patients_raw
    WHERE file_id = :file_id
      AND batch_id = :batch_id
      AND adf_run_id = :adf_run_id
""", args={
    "file_id": file_id,
    "batch_id": batch_id,
    "adf_run_id": adf_run_id
}).first()

bronze_count = bronze_stats["bronze_count"]
unique_patients = bronze_stats["unique_patients"]
invalid_patient_ids = bronze_stats["invalid_patient_ids"]

if bronze_count == 0:
    raise RuntimeError(
        "Silver processing rejected: No Bronze records found"
    )

if invalid_patient_ids > 0:
    raise RuntimeError(
        f"Invalid patient IDs: {invalid_patient_ids}"
    )

if bronze_count != unique_patients:
    raise RuntimeError(
        "Duplicate patient IDs detected in Bronze batch"
    )

print("Bronze source validation passed")
print("Bronze records:", bronze_count)
print("Unique patients:", unique_patients)
print("Invalid patient IDs:", invalid_patient_ids)


In [0]:

import uuid

silver_audit_id = str(uuid.uuid4())

spark.sql("""
    CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit(
        'START',
        :audit_id,
        :file_id,
        :adf_run_id,
        'SILVER',
        NULL,
        NULL,
        NULL,
        NULL
    )
""", args={
    "audit_id": silver_audit_id,
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

# Verify audit was actually created
audit_rows = spark.sql("""
    SELECT stage_status
    FROM dbw_healthcare_de_dev.etl.file_stage_audit
    WHERE audit_id = :audit_id
      AND file_id = :file_id
      AND adf_run_id = :adf_run_id
      AND processing_stage = 'SILVER'
""", args={
    "audit_id": silver_audit_id,
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

if len(audit_rows) != 1 or audit_rows[0]["stage_status"] != "IN_PROGRESS":
    raise RuntimeError("Silver audit START could not be verified")

print("Silver audit started:", silver_audit_id)


In [0]:

# Do not automatically retry uncertain failures.

try:
    result = spark.sql("""
        CALL dbw_healthcare_de_dev.silver.usp_LoadDimPatients(
            :file_id
        )
    """, args={
        "file_id": file_id
    }).collect()

    print("Silver stored procedure completed")

    for row in result:
        print(row)

except Exception as error:

    print("SILVER PROCESSING FAILED")
    print("File ID:", file_id)
    print("Audit ID:", silver_audit_id)
    print("Error:", str(error))

    # The procedure may have partially committed.
    # Preserve the IN_PROGRESS audit for reconciliation.
    # Do not mark SUCCESS or advance to GOLD.

    raise RuntimeError(
        f"Silver load failed for file_id={file_id}. "
        f"Audit ID={silver_audit_id}. "
        "Manual reconciliation is required before retrying."
    ) from error


In [0]:

validation = spark.sql("""
    SELECT
        COUNT(*) AS source_count,
        SUM(
            CASE
                WHEN D.PatientId IS NULL THEN 1
                WHEN NOT (B.first_name <=> D.FirstName) THEN 1
                WHEN NOT (B.last_name <=> D.LastName) THEN 1
                WHEN NOT (B.gender <=> D.Gender) THEN 1
                WHEN NOT (B.date_of_birth <=> D.DateOfBirth) THEN 1
                WHEN NOT (B.state <=> D.State) THEN 1
                ELSE 0
            END
        ) AS mismatched_records
    FROM dbw_healthcare_de_dev.bronze.patients_raw B
    LEFT JOIN dbw_healthcare_de_dev.silver.DimPatients D
        ON B.patient_id = D.PatientId
       AND D.IsActive = TRUE
    WHERE B.file_id = :file_id
      AND B.batch_id = :batch_id
      AND B.adf_run_id = :adf_run_id
""", args={
    "file_id": file_id,
    "batch_id": batch_id,
    "adf_run_id": adf_run_id
}).first()

if (
    validation["source_count"] != bronze_count
    or validation["mismatched_records"] != 0
):
    raise RuntimeError(
        "Silver validation failed: source-to-target mismatch"
    )

print("Silver source-to-target validation passed")
print("Validated records:", validation["source_count"])


In [0]:


scd_validation = spark.sql("""
    SELECT
        B.patient_id,
        COUNT(D.DimPatientID) AS active_versions
    FROM (
        SELECT DISTINCT patient_id
        FROM dbw_healthcare_de_dev.bronze.patients_raw
        WHERE file_id = :file_id
          AND batch_id = :batch_id
          AND adf_run_id = :adf_run_id
    ) B
    LEFT JOIN dbw_healthcare_de_dev.silver.DimPatients D
        ON B.patient_id = D.PatientId
       AND D.IsActive = TRUE
    GROUP BY B.patient_id
    HAVING COUNT(D.DimPatientID) <> 1
""", args={
    "file_id": file_id,
    "batch_id": batch_id,
    "adf_run_id": adf_run_id
}).collect()

if scd_validation:
    raise RuntimeError(
        f"Invalid active SCD versions: {scd_validation}"
    )

print("SCD validation passed")
print("Every incoming patient has exactly one active record")


In [0]:

spark.sql("""
    CALL dbw_healthcare_de_dev.etl.usp_manage_stage_audit(
        'SUCCESS',
        :audit_id,
        :file_id,
        :adf_run_id,
        'SILVER',
        :source_count,
        :processed_count,
        0,
        NULL
    )
""", args={
    "audit_id": silver_audit_id,
    "file_id": file_id,
    "adf_run_id": adf_run_id,
    "source_count": bronze_count,
    "processed_count": bronze_count
}).collect()

audit_result = spark.sql("""
    SELECT stage_status, stage_end_ts
    FROM dbw_healthcare_de_dev.etl.file_stage_audit
    WHERE audit_id = :audit_id
      AND file_id = :file_id
      AND adf_run_id = :adf_run_id
      AND processing_stage = 'SILVER'
""", args={
    "audit_id": silver_audit_id,
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).first()

if (
    audit_result is None
    or audit_result["stage_status"] != "SUCCESS"
    or audit_result["stage_end_ts"] is None
):
    raise RuntimeError(
        "Silver audit SUCCESS could not be verified"
    )

print("Silver audit completed successfully")


In [0]:

spark.sql("""
    CALL dbw_healthcare_de_dev.etl.usp_update_file_stage(
        :file_id,
        :adf_run_id,
        'SILVER'
    )
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

stage_result = spark.sql("""
    SELECT processing_stage, pipeline_status
    FROM dbw_healthcare_de_dev.etl.file_control
    WHERE file_id = :file_id
      AND adf_run_id = :adf_run_id
""", args={
    "file_id": file_id,
    "adf_run_id": adf_run_id
}).collect()

if (
    len(stage_result) != 1
    or stage_result[0]["processing_stage"] != "GOLD"
    or stage_result[0]["pipeline_status"] != "IN_PROGRESS"
):
    raise RuntimeError(
        "Silver-to-Gold transition could not be verified"
    )

print("Silver processing completed")
print("File successfully advanced to GOLD")
